### First, we install Librosa's library into Google collab's terminal
```
pip install librosa
```
Then import the library in the code.

In [2]:
import librosa as li
print(li.__version__)

0.11.0


### Downloading the dataset from kaggle
Import kagglehub and pathlib to download the dataset

In [3]:
import kagglehub
from pathlib import Path

genres_original_path = next(Path(kagglehub.dataset_download(
    "andradaolteanu/gtzan-dataset-music-genre-classification")).rglob("genres_original"))

Using Colab cache for faster access to the 'gtzan-dataset-music-genre-classification' dataset.


In [ ]:
import warnings
import numpy as np
import pandas as pd
import librosa as li

rows = []
audio_extensions = {".wav", ".mp3", ".au", ".flac", ".ogg", ".m4a"}

for genre_dir in sorted(genres_original_path.iterdir()):
    if not genre_dir.is_dir():
        continue

    label = genre_dir.name

    for song_path in sorted(genre_dir.iterdir()):
        if song_path.suffix.lower() not in audio_extensions:
            continue

        try:
            # Load audio at a fixed sample rate so all songs are comparable.
            with warnings.catch_warnings():
                warnings.filterwarnings("ignore", message="PySoundFile failed.*")
                warnings.filterwarnings("ignore", message="librosa.core.audio.__audioread_load.*")
                y, sr = li.load(song_path, sr=22050, mono=True)

            mfcc = li.feature.mfcc(y=y, sr=sr, n_mfcc=20)
            chroma = li.feature.chroma_stft(y=y, sr=sr)
            spectral_centroid = li.feature.spectral_centroid(y=y, sr=sr)
            spectral_bandwidth = li.feature.spectral_bandwidth(y=y, sr=sr)
            spectral_rolloff = li.feature.spectral_rolloff(y=y, sr=sr)
            spectral_contrast = li.feature.spectral_contrast(y=y, sr=sr)
            tonnetz = li.feature.tonnetz(y=li.effects.harmonic(y), sr=sr)
            rms = li.feature.rms(y=y)
            zcr = li.feature.zero_crossing_rate(y)
            tempo = float(np.asarray(li.beat.beat_track(y=y, sr=sr)[0]).reshape(-1)[0])

            rows.append({
                "mfcc_mean": float(np.mean(mfcc)),
                "mfcc_std": float(np.std(mfcc)),
                "chroma_mean": float(np.mean(chroma)),
                "chroma_std": float(np.std(chroma)),
                "spectral_centroid_mean": float(np.mean(spectral_centroid)),
                "spectral_centroid_std": float(np.std(spectral_centroid)),
                "spectral_bandwidth_mean": float(np.mean(spectral_bandwidth)),
                "spectral_bandwidth_std": float(np.std(spectral_bandwidth)),
                "spectral_rolloff_mean": float(np.mean(spectral_rolloff)),
                "spectral_rolloff_std": float(np.std(spectral_rolloff)),
                "spectral_contrast_mean": float(np.mean(spectral_contrast)),
                "spectral_contrast_std": float(np.std(spectral_contrast)),
                "tonnetz_mean": float(np.mean(tonnetz)),
                "tonnetz_std": float(np.std(tonnetz)),
                "rms_mean": float(np.mean(rms)),
                "rms_std": float(np.std(rms)),
                "zero_crossing_rate_mean": float(np.mean(zcr)),
                "zero_crossing_rate_std": float(np.std(zcr)),
                "tempo": tempo,
                "label": label,
            })
        except Exception as exc:
            print(f"Skipping {song_path.name} ({label}): {exc}")

features_df = pd.DataFrame(rows)
output_csv = Path.cwd() / "gtzan_selected_features.csv"
features_df.to_csv(output_csv, index=False)

print(f"CSV created successfully: {output_csv}")
print(f"Total songs processed: {len(features_df)}")
features_df.head()

Skipping jazz.00054.wav (jazz): 
CSV created successfully: /content/gtzan_selected_features.csv
Total songs processed: 999


,mfcc_std,chroma_mean,spectral_centroid_mean,spectral_rolloff_mean,zero_crossing_rate_mean,tempo,label
0,42.042160,0.350129,1784.122641,3805.723030,0.083045,123.046875,blues
1,60.053635,0.340849,1530.261767,3550.713616,0.056040,67.999589,blues
2,43.078026,0.363538,1552.832481,3042.410115,0.076291,161.499023,blues
3,59.700958,0.404854,1070.153418,2184.879029,0.033309,63.024009,blues
4,51.301071,0.308526,1835.128513,3579.957471,0.101461,135.999178,blues


In [5]:
from pathlib import Path
from IPython.display import FileLink, display

csv_path = Path.cwd() / "gtzan_selected_features.csv"

if csv_path.exists():
    display(FileLink(str(csv_path)))
    try:
        from google.colab import files
        files.download(str(csv_path))
    except Exception:
        print(f"CSV ready at: {csv_path}")
else:
    print("Run Cell 5 first to create the CSV.")

/content/gtzan_selected_features.csv

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>